In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS iot_sensor_anomoly_detection.gold;

In [0]:
device_health_df = spark.table("iot_sensor_anomoly_detection.silver.device_health_diagnostics_clean")
device_ops_df = spark.table("iot_sensor_anomoly_detection.silver.device_operations_clean")
env_network_df = spark.table("iot_sensor_anomoly_detection.silver.environment_network_clean")
sensor_stream_df = spark.table("iot_sensor_anomoly_detection.silver.sensor_stream_clean")
anomaly_events_df = spark.table("iot_sensor_anomoly_detection.silver.time_anomaly_events_clean")

Creating Dim_Device table

In [0]:
dim_device = device_ops_df.select(
    "device_id",
    "device_category",
    "maker_name",
    "firmware_build",
    "production_line",
    "factory_site"
).dropDuplicates(["device_id"])

dim_device.write.format("delta") \
.mode("overwrite") \
.saveAsTable("iot_sensor_anomoly_detection.gold.dim_device")

In [0]:
%sql
SELECT *
FROM iot_sensor_anomoly_detection.gold.dim_device;

Creating Dim_Sensor

In [0]:
dim_sensor = sensor_stream_df.select(
    "sensor_identifier",
    "sensor_category",
    "sensor_model",
    "sensor_zone",
    "reading_unit"
).dropDuplicates(["sensor_identifier"])

dim_sensor.write.format("delta") \
.mode("overwrite") \
.saveAsTable("iot_sensor_anomoly_detection.gold.dim_sensor")

In [0]:
%sql
SELECT *
FROM iot_sensor_anomoly_detection.gold.dim_sensor limit 40000;


Creating Dim_Location

In [0]:
dim_location = env_network_df.select(
    "device_id",
    "region_name",
    "city_name",
    "site_location"
).dropDuplicates(["device_id"])

dim_location.write.format("delta") \
.mode("overwrite") \
.saveAsTable("iot_sensor_anomoly_detection.gold.dim_location")

In [0]:
%sql
SELECT *
FROM iot_sensor_anomoly_detection.gold.dim_location limit 100;

Creating Dim_Time table

In [0]:
from pyspark.sql.functions import year, month, dayofmonth, hour, dayofweek

dim_time = anomaly_events_df.select(
    anomaly_events_df.event_timestamp.alias("timestamp")
).dropDuplicates(["timestamp"]) \
.withColumn("year", year("timestamp")) \
.withColumn("month", month("timestamp")) \
.withColumn("day", dayofmonth("timestamp")) \
.withColumn("hour", hour("timestamp")) \
.withColumn("day_of_week", dayofweek("timestamp"))

dim_time.write.format("delta") \
.mode("overwrite") \
.saveAsTable("iot_sensor_anomoly_detection.gold.dim_time")

In [0]:
%sql
SELECT *
FROM iot_sensor_anomoly_detection.gold.dim_time limit 100;

#### final fact table code

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

# Silver tables
device_health_df = spark.table("iot_sensor_anomoly_detection.silver.device_health_diagnostics_clean")
device_ops_df = spark.table("iot_sensor_anomoly_detection.silver.device_operations_clean")
env_network_df = spark.table("iot_sensor_anomoly_detection.silver.environment_network_clean")
sensor_stream_df = spark.table("iot_sensor_anomoly_detection.silver.sensor_stream_clean")
anomaly_events_df = spark.table("iot_sensor_anomoly_detection.silver.time_anomaly_events_clean")

# -------------------------------------------------
# Join sensor data with events to get timestamp
# -------------------------------------------------

sensor_events = sensor_stream_df.alias("s") \
.join(anomaly_events_df.alias("a"), "device_id", "left")

# -------------------------------------------------
# Rolling Window
# -------------------------------------------------

windowSpec = Window.partitionBy("device_id") \
.orderBy("event_timestamp") \
.rowsBetween(-5,0)

sensor_metrics = sensor_events \
.withColumn(
    "rolling_avg_reading",
    avg("reading_value").over(windowSpec)
)

# -------------------------------------------------
# Z-score calculation
# -------------------------------------------------

stats = sensor_stream_df.select(
    mean("reading_value").alias("mean"),
    stddev("reading_value").alias("std")
).collect()[0]

mean_val = stats["mean"]
std_val = stats["std"]

sensor_metrics = sensor_metrics.withColumn(
    "z_score",
    (col("reading_value") - mean_val) / std_val
)

# -------------------------------------------------
# Status tagging
# -------------------------------------------------

sensor_metrics = sensor_metrics.withColumn(
    "sensor_status",
    when(col("z_score") > 2, "CRITICAL")
    .when(col("z_score") > 1, "WARNING")
    .otherwise("NORMAL")
)

# -------------------------------------------------
# Final Fact Table
# -------------------------------------------------

fact_device_monitoring = sensor_metrics.alias("s") \
.join(device_health_df.alias("h"), "device_id", "left") \
.join(env_network_df.alias("e"), "device_id", "left") \
.select(
    col("s.device_id"),
    col("s.sensor_identifier"),
    col("s.event_timestamp"),
    col("s.reading_value"),
    col("s.rolling_avg_reading"),
    col("s.z_score"),
    col("s.sensor_status"),
    col("h.health_score"),
    col("h.battery_voltage"),
    col("h.error_count"),
    col("h.restart_count"),
    col("s.anomaly_category"),
    col("s.severity_level"),
    col("s.downtime_minutes"),
    col("e.region_name"),
    col("e.network_provider")
)

fact_device_monitoring.write.format("delta") \
.mode("overwrite") \
.saveAsTable("iot_sensor_anomoly_detection.gold.fact_device_monitoring")

In [0]:
%sql
SELECT *
FROM iot_sensor_anomoly_detection.gold.fact_device_monitoring;

#### Insights

In [0]:
from pyspark.sql.functions import *

fact_df = spark.table("iot_sensor_anomoly_detection.gold.fact_device_monitoring")
dim_device = spark.table("iot_sensor_anomoly_detection.gold.dim_device")
dim_sensor = spark.table("iot_sensor_anomoly_detection.gold.dim_sensor")
dim_location = spark.table("iot_sensor_anomoly_detection.gold.dim_location")
dim_time = spark.table("iot_sensor_anomoly_detection.gold.dim_time")

#### Low Level Insights

#### Sensor Status Distribution

In [0]:
sensor_status_dist = fact_df.groupBy("sensor_status") \
.agg(count("*").alias("total_records"))

sensor_status_dist.show()

#### Devices with Most Errors

In [0]:
device_errors = fact_df.groupBy("device_id") \
.agg(sum("error_count").alias("total_errors")) \
.orderBy(col("total_errors").desc())

device_errors.show()

#### Average Battery Voltage by Region

In [0]:
battery_region = fact_df.groupBy("region_name") \
.agg(avg("battery_voltage").alias("avg_battery_voltage"))

battery_region.show()

#### Medium Level Insights


#### Most Problematic Sensors

In [0]:
sensor_failures = fact_df \
.filter(col("sensor_status") != "NORMAL") \
.groupBy("sensor_identifier") \
.agg(count("*").alias("anomaly_count")) \
.orderBy(col("anomaly_count").desc())

sensor_failures.show()

#### Network Provider Impact on Device Issues

In [0]:
network_issues = fact_df.groupBy("network_provider") \
.agg(
count("*").alias("total_records"),
sum(when(col("sensor_status")!="NORMAL",1).otherwise(0)).alias("anomaly_records")
)

network_issues.show()

#### Average sensor Reading by Sensor Type

In [0]:
sensor_performance = fact_df \
.groupBy("sensor_identifier") \
.agg(
avg("reading_value").alias("avg_reading"),
avg("rolling_avg_reading").alias("avg_rolling_reading")
)

sensor_performance.show()

#### Advanced Insights

#### Device Health vs Anomalies

In [0]:
health_vs_anomaly = fact_df \
.groupBy("device_id") \
.agg(
avg("health_score").alias("avg_health"),
sum(when(col("sensor_status")!="NORMAL",1).otherwise(0)).alias("total_anomalies")
) \
.orderBy(col("total_anomalies").desc())

health_vs_anomaly.show()

#### Downtime Analysis

In [0]:
downtime_analysis = fact_df \
.groupBy("device_id") \
.agg(sum("downtime_minutes").alias("total_downtime")) \
.orderBy(col("total_downtime").desc())

downtime_analysis.show()

#### Region-wise Anomaly Detection

In [0]:
region_anomalies = fact_df \
.filter(col("sensor_status")!="NORMAL") \
.groupBy("region_name") \
.agg(count("*").alias("total_anomalies")) \
.orderBy(col("total_anomalies").desc())

region_anomalies.show()

#### Critical Device Monitoring

In [0]:
critical_devices = fact_df \
.filter(col("sensor_status")=="CRITICAL") \
.select("device_id","sensor_identifier","reading_value","z_score","region_name")

critical_devices.show()